# Warden v2: do motor de regras ao controlador fuzzy

Esta e a versao 2 do [Warden](cyber_warden.ipynb): as celulas abaixo, ate
"Autoverificacao", sao o motor de regras original (mesmos Facts, mesmo motor de
encadeamento progressivo, mesmo cenario de demonstracao) com dois ajustes:

- **Regra nova `ameaca_integridade`.** Em v1, um arquivo critico alterado sem
  nenhum outro sinal gerava so um `Rastro` e nenhuma decisao: o caso mais
  grave passava em silencio. Agora ele levanta uma ameaca de nivel alto
  (`MONITORAR`), e so escala para `ISOLAR` quando outra ameaca o corrobora.
- **Saliences em camadas** (indicadores 120, ameacas 90), para que todos os
  indicadores e ameacas existam antes de qualquer decisao. Sem isso, a nova
  regra deixava `decisao_monitorar` disparar cedo demais num host que ainda ia
  escalar.

A partir de "Controlador fuzzy Mamdani", o notebook acrescenta uma camada
nova: em vez de so uma decisao crisp (`ISOLAR`/`MONITORAR`/nenhuma), a mesma
evidencia bruta de cada host — as mesmas `Conexao` e `Arquivo` da secao
acima, com os mesmos limiares `EXPEDIENTE` e `LIMITE_EXFILTRACAO` — alimenta
um controlador fuzzy Mamdani que produz uma prioridade continua de 0 a 10.
As duas leituras (crisp e fuzzy) sao comparadas lado a lado no final.

Encadeamento progressivo em cadeia: **evidencia -> indicador -> ameaca -> decisao**.

A Base de Conhecimento (BC) sao as classes `Fact` e as regras `@Rule`. A Memoria de Trabalho (MT) e alimentada pelos `declare`. O motor de inferencia foi reescrito para expor a agenda (conjunto-conflito) a cada ciclo, e a refracao (refractoriness) do experta impede que a mesma ativacao dispare duas vezes com os mesmos fatos.

In [1]:
!pip install -q experta


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Patch de compatibilidade e base de conhecimento

O `frozendict 1.2`, dependencia do experta, usa `collections.Mapping`, removido no Python 3.10.

Fixar `frozendict==1.2` nao resolve: e justamente essa versao que quebra.

In [2]:
"""Warden: sistema especialista de seguranca cibernetica.

Encadeamento progressivo em cadeia:
    evidencia -> indicador (rastro) -> ameaca -> decisao
"""

# frozendict 1.2 (dependencia do experta) 
import collections
import collections.abc
import sys

for _nome in ("Mapping", "MutableMapping", "Sequence", "Iterable", "Set", "Hashable"):
    if not hasattr(collections, _nome):
        setattr(collections, _nome, getattr(collections.abc, _nome))

from experta import AS, MATCH, NOT, L, P, Fact, KnowledgeEngine, Rule

## Base de Conhecimento: definicao dos fatos

In [ ]:
class Conexao(Fact):
    """Evidencia de rede observada para um host. Nivel 0 da cadeia.

    origem: str           -- IP de origem
    portas: int           -- portas distintas tocadas na janela
    tentativas_login: int -- falhas de autenticacao na janela
    bytes_saida: int      -- volume enviado para fora, em bytes
    hora: int             -- hora do dia, 0 a 23
    """


class Arquivo(Fact):
    """Evidencia de integridade de arquivo. Nivel 0 da cadeia.

    caminho: str    -- caminho absoluto do arquivo
    origem: str     -- IP associado a alteracao
    critico: bool   -- arquivo faz parte da base do sistema
    hash_mudou: bool -- checksum divergiu do baseline
    """


class Rastro(Fact):
    """Indicador derivado de uma evidencia. Nivel 1 da cadeia.

    nome: str   -- forca_bruta | varredura | exfiltracao | integridade_violada
    origem: str -- IP responsavel
    motivo: str -- justificativa legivel, usada na explicacao
    """


class Ameaca(Fact):
    """Hipotese de ameaca sustentada por indicadores. Nivel 2 da cadeia.

    nome: str   -- intrusao_ativa | vazamento_dados | integridade_comprometida | comprometimento_raiz
    origem: str -- IP responsavel
    nivel: str  -- alto | critico
    """


class Decisao(Fact):
    """Acao recomendada pelo sistema. Nivel 3 da cadeia.

    acao: str   -- ISOLAR | MONITORAR
    alvo: str   -- IP alvo da acao
    porque: str -- justificativa legivel
    """

## Motor de inferencia + subsistema de explicacao

In [ ]:
EXPEDIENTE = range(8, 19)
LIMITE_EXFILTRACAO = 500_000_000


class Warden(KnowledgeEngine):
    """Motor de encadeamento progressivo com agenda visivel."""

    def __init__(self):
        super().__init__()
        self.disparos = []

    def _registrar(self, origem, motivo):
        # O nome da regra vem do frame que chamou, e nao de uma string repetida
        # em cada regra, que silenciosamente mentiria apos um rename.
        regra = sys._getframe(1).f_code.co_name
        self.disparos.append((regra, origem, motivo))

    def explicar(self):
        """Trilha de raciocinio agrupada pela decisao que ela sustenta.

        A ordem cronologica crua fica em self.disparos. Aqui ela e reagrupada
        por host, porque a pergunta que o subsistema de explicacao precisa
        responder e "por que esta decisao", e nao "o que aconteceu primeiro".
        """
        acoes = {d["alvo"]: d["acao"] for d in self.decisoes()}
        alvos = dict.fromkeys(origem for _, origem, _ in self.disparos)

        blocos = []
        for alvo in alvos:
            passos = [
                f"  {i}. [{regra}] {motivo}"
                for i, (regra, motivo) in enumerate(
                    ((r, m) for r, o, m in self.disparos if o == alvo), 1
                )
            ]
            cabecalho = f"{acoes.get(alvo, 'SEM DECISAO')} {alvo}"
            blocos.append("\n".join([cabecalho, *passos]))

        return "\n\n".join(blocos)

    def decisoes(self):
        return [f for f in self.facts.values() if isinstance(f, Decisao)]

    def run(self, steps=float("inf"), verboso=False):
        """Mesmo laco do KnowledgeEngine.run, com a agenda impressa a cada ciclo."""
        self.running = True
        ciclo = 0
        while steps > 0 and self.running:
            # 1. Casamento de padroes: atualiza o conjunto-conflito.
            adicionadas, removidas = self.get_activations()
            self.strategy.update_agenda(self.agenda, adicionadas, removidas)
            ciclo += 1

            if verboso:
                print(f"\n--- ciclo {ciclo} | agenda (conjunto-conflito) ---")
                if not self.agenda.activations:
                    print("  (vazia)")
                for i, act in enumerate(self.agenda.activations):
                    print(f"  {i}: {act.rule.__name__}")

            # 2. Agenda vazia encerra a execucao.
            ativacao = self.agenda.get_next()
            if ativacao is None:
                break

            # 3. Resolucao de conflito: a estrategia ja ordenou a agenda por
            #    salience, e a refracao impede o mesmo disparo duas vezes.
            # 4. Execucao do lado direito da regra escolhida.
            steps -= 1
            if verboso:
                print(f"  -> dispara {ativacao.rule.__name__}")
            ativacao.rule(
                self,
                **{
                    k: v
                    for k, v in ativacao.context.items()
                    if not k.startswith("__")
                },
            )
            # 5. Volta ao passo 1 com a memoria de trabalho ja alterada.

        self.running = False

    # Saliences em camadas, para a agenda nunca decidir antes de a cadeia fechar:
    #   120 indicadores > 110 revogar > 100 isolar > 90 ameacas > 80 critica > 50 monitorar
    @Rule(AS.c << Conexao(origem=MATCH.o, tentativas_login=P(lambda t: t >= 10)), salience=120)
    def indicador_forca_bruta(self, c, o):
        motivo = f"{c['tentativas_login']} falhas de login vindas de {o}"
        self.declare(Rastro(nome="forca_bruta", origem=o, motivo=motivo))
        self._registrar(o, motivo)

    @Rule(AS.c << Conexao(origem=MATCH.o, portas=P(lambda p: p >= 100)), salience=120)
    def indicador_varredura(self, c, o):
        motivo = f"{c['portas']} portas distintas tocadas por {o}"
        self.declare(Rastro(nome="varredura", origem=o, motivo=motivo))
        self._registrar(o, motivo)

    @Rule(
        AS.c
        << Conexao(
            origem=MATCH.o,
            bytes_saida=P(lambda b: b > LIMITE_EXFILTRACAO),
            hora=P(lambda h: h not in EXPEDIENTE),
        ),
        salience=120,
    )
    def indicador_exfiltracao(self, c, o):
        mb = c["bytes_saida"] // 1_000_000
        motivo = f"{mb} MB enviados por {o} as {c['hora']}h, fora do expediente"
        self.declare(Rastro(nome="exfiltracao", origem=o, motivo=motivo))
        self._registrar(o, motivo)

    @Rule(AS.a << Arquivo(origem=MATCH.o, critico=True, hash_mudou=True), salience=120)
    def indicador_integridade(self, a, o):
        motivo = f"checksum de {a['caminho']} divergiu, alteracao ligada a {o}"
        self.declare(Rastro(nome="integridade_violada", origem=o, motivo=motivo))
        self._registrar(o, motivo)

    @Rule(
        Rastro(nome="varredura", origem=MATCH.o),
        Rastro(nome="forca_bruta", origem=MATCH.o),
        salience=90,
    )
    def ameaca_intrusao(self, o):
        motivo = f"varredura e forca bruta partindo do mesmo host {o}"
        self.declare(Ameaca(nome="intrusao_ativa", origem=o, nivel="alto"))
        self._registrar(o, motivo)

    @Rule(Rastro(nome="exfiltracao", origem=MATCH.o), salience=90)
    def ameaca_vazamento(self, o):
        motivo = f"volume anormal de saida atribuido a {o}"
        self.declare(Ameaca(nome="vazamento_dados", origem=o, nivel="alto"))
        self._registrar(o, motivo)

    @Rule(Rastro(nome="integridade_violada", origem=MATCH.o), salience=90)
    def ameaca_integridade(self, o):
        motivo = f"integridade violada em {o}, sem outro indicador corroborando"
        self.declare(Ameaca(nome="integridade_comprometida", origem=o, nivel="alto"))
        self._registrar(o, motivo)

    # integridade_comprometida e a propria integridade violada: nao conta como a
    # "outra" ameaca que ameaca_critica exige, senao ela escalaria sozinha.
    @Rule(
        Rastro(nome="integridade_violada", origem=MATCH.o),
        Ameaca(nome=MATCH.t1 & ~L("integridade_comprometida"), origem=MATCH.o, nivel="alto"),
        salience=80,
    )
    def ameaca_critica(self, o, t1):
        motivo = f"integridade violada somada a {t1} no host {o}"
        self.declare(Ameaca(nome="comprometimento_raiz", origem=o, nivel="critico"))
        self._registrar(o, motivo)

    # ISOLAR e MONITORAR sao mutuamente exclusivas para um mesmo host. Tres
    # mecanismos, nessa ordem, garantem isso:
    #   salience   -- ordena o conjunto-conflito quando as duas estao na agenda;
    #   NOT(...)   -- tira decisao_monitorar da agenda assim que a hipotese
    #                 critica entra na MT;
    #   retract    -- revoga um MONITORAR ja declarado numa rodada anterior,
    #                 que o NOT sozinho nao alcanca por so barrar ativacao nova.

    @Rule(
        Ameaca(origem=MATCH.o, nivel="critico"),
        AS.d << Decisao(acao="MONITORAR", alvo=MATCH.o),
        salience=110,
    )
    def revogar_monitoramento(self, d, o):
        motivo = f"monitoramento de {o} revogado: a ameaca escalou para critica"
        self.retract(d)
        self._registrar(o, motivo)

    @Rule(Ameaca(nome=MATCH.n, origem=MATCH.o, nivel="critico"), salience=100)
    def decisao_isolar(self, n, o):
        motivo = f"ameaca critica {n} confirmada em {o}"
        self.declare(Decisao(acao="ISOLAR", alvo=o, porque=motivo))
        self._registrar(o, motivo)

    @Rule(
        Ameaca(nome=MATCH.n, origem=MATCH.o, nivel="alto"),
        NOT(Ameaca(origem=MATCH.o, nivel="critico")),
        salience=50,
    )
    def decisao_monitorar(self, n, o):
        motivo = f"ameaca {n} em {o} sem evidencia de comprometimento de raiz"
        self.declare(Decisao(acao="MONITORAR", alvo=o, porque=motivo))
        self._registrar(o, motivo)

## Cenario da memoria de trabalho

In [5]:
def _acoes_por_alvo(motor):
    return {d["alvo"]: d["acao"] for d in motor.decisoes()}


def demo(verboso=False):
    motor = Warden()
    motor.reset()

    # Atacante: varre portas, tenta forca bruta e altera arquivo de sistema.
    motor.declare(
        Conexao(
            origem="10.0.0.66",
            portas=512,
            tentativas_login=47,
            bytes_saida=1_000,
            hora=3,
        )
    )
    motor.declare(
        Arquivo(
            caminho="/etc/shadow",
            origem="10.0.0.66",
            critico=True,
            hash_mudou=True,
        )
    )
    # Suspeito: so exfiltracao, sem comprometimento de raiz.
    motor.declare(
        Conexao(
            origem="10.0.0.20",
            portas=3,
            tentativas_login=0,
            bytes_saida=900_000_000,
            hora=2,
        )
    )
    # Benigno: nada acima dos limiares.
    motor.declare(
        Conexao(
            origem="10.0.0.5",
            portas=2,
            tentativas_login=1,
            bytes_saida=1_000,
            hora=14,
        )
    )

    motor.run(verboso=verboso)
    return motor

## Checagens executaveis

In [ ]:
def _rastros(**campos):
    """Nomes dos indicadores que uma unica Conexao produz."""
    motor = Warden()
    motor.reset()
    motor.declare(Conexao(origem="1.1.1.1", **campos))
    motor.run()
    return {f["nome"] for f in motor.facts.values() if isinstance(f, Rastro)}


def _verificar_cadeia():
    motor = demo()
    acoes = _acoes_por_alvo(motor)

    assert acoes["10.0.0.66"] == "ISOLAR", acoes
    assert acoes["10.0.0.20"] == "MONITORAR", acoes
    assert "10.0.0.5" not in acoes, acoes
    assert motor.disparos, "subsistema de explicacao vazio"

    # A salience mantem ameaca_critica (80) a frente de decisao_monitorar (50),
    # entao o host critico nunca chega a receber uma recomendacao de
    # monitoramento, nem provisoria.
    assert not any(
        regra == "decisao_monitorar" and "10.0.0.66" in motivo
        for regra, _, motivo in motor.disparos
    ), motor.explicar()


def _verificar_limiares():
    base = dict(portas=1, tentativas_login=0, bytes_saida=0, hora=3)

    # Cada limiar e verificado dos dois lados, entao afrouxa-lo ou aperta-lo
    # muda o resultado e quebra a checagem.
    assert "forca_bruta" not in _rastros(**{**base, "tentativas_login": 9})
    assert "forca_bruta" in _rastros(**{**base, "tentativas_login": 10})

    assert "varredura" not in _rastros(**{**base, "portas": 99})
    assert "varredura" in _rastros(**{**base, "portas": 100})

    volumoso = {**base, "bytes_saida": LIMITE_EXFILTRACAO + 1}
    assert "exfiltracao" not in _rastros(**{**base, "bytes_saida": LIMITE_EXFILTRACAO})
    assert "exfiltracao" in _rastros(**volumoso)
    # O mesmo volume dentro do expediente nao e exfiltracao.
    assert "exfiltracao" not in _rastros(**{**volumoso, "hora": 14})


def _verificar_exclusao_mutua():
    """A escalada pode chegar numa rodada posterior, com MONITORAR ja na MT."""
    motor = Warden()
    motor.reset()
    motor.declare(
        Conexao(
            origem="2.2.2.2",
            portas=512,
            tentativas_login=47,
            bytes_saida=1_000,
            hora=3,
        )
    )
    motor.run()
    assert _acoes_por_alvo(motor) == {"2.2.2.2": "MONITORAR"}, motor.decisoes()

    motor.declare(
        Arquivo(
            caminho="/etc/shadow",
            origem="2.2.2.2",
            critico=True,
            hash_mudou=True,
        )
    )
    motor.run()
    assert _acoes_por_alvo(motor) == {"2.2.2.2": "ISOLAR"}, motor.decisoes()
    assert len(motor.decisoes()) == 1, motor.decisoes()

    # A revogacao (salience 110) vem antes da nova decisao (100).
    regras = [regra for regra, _, _ in motor.disparos]
    assert regras.index("revogar_monitoramento") < regras.index("decisao_isolar"), regras


def _verificar_integridade_isolada():
    """Arquivo critico alterado, sem nenhum outro sinal: nem silencio, nem ISOLAR."""
    motor = Warden()
    motor.reset()
    motor.declare(
        Arquivo(caminho="/etc/shadow", origem="3.3.3.3", critico=True, hash_mudou=True)
    )
    motor.run()
    assert _acoes_por_alvo(motor) == {"3.3.3.3": "MONITORAR"}, motor.decisoes()


def _autoverificar():
    _verificar_cadeia()
    _verificar_limiares()
    _verificar_integridade_isolada()
    _verificar_exclusao_mutua()
    print("autoverificacao ok")

## Execucao: agenda e conjunto-conflito visiveis

In [7]:
motor = demo(verboso=True)


--- ciclo 1 | agenda (conjunto-conflito) ---
  0: indicador_forca_bruta
  1: indicador_varredura
  2: indicador_integridade
  3: indicador_exfiltracao
  -> dispara indicador_exfiltracao

--- ciclo 2 | agenda (conjunto-conflito) ---
  0: ameaca_vazamento
  1: indicador_forca_bruta
  2: indicador_varredura
  3: indicador_integridade
  -> dispara indicador_integridade

--- ciclo 3 | agenda (conjunto-conflito) ---
  0: ameaca_vazamento
  1: ameaca_integridade
  2: indicador_forca_bruta
  3: indicador_varredura
  -> dispara indicador_varredura

--- ciclo 4 | agenda (conjunto-conflito) ---
  0: ameaca_vazamento
  1: ameaca_integridade
  2: indicador_forca_bruta
  -> dispara indicador_forca_bruta

--- ciclo 5 | agenda (conjunto-conflito) ---
  0: ameaca_vazamento
  1: ameaca_integridade
  2: ameaca_intrusao
  -> dispara ameaca_intrusao

--- ciclo 6 | agenda (conjunto-conflito) ---
  0: decisao_monitorar
  1: ameaca_critica
  2: ameaca_vazamento
  3: ameaca_integridade
  -> dispara ameaca_int

## Decisoes

In [8]:
for d in motor.decisoes():
    print(f"{d['acao']:<10} {d['alvo']:<12} {d['porque']}")

ISOLAR     10.0.0.66    ameaca critica comprometimento_raiz confirmada em 10.0.0.66
MONITORAR  10.0.0.20    ameaca vazamento_dados em 10.0.0.20 sem evidencia de comprometimento de raiz


## Subsistema de explicacao

In [9]:
print(motor.explicar())

MONITORAR 10.0.0.20
  1. [indicador_exfiltracao] 900 MB enviados por 10.0.0.20 as 2h, fora do expediente
  2. [ameaca_vazamento] volume anormal de saida atribuido a 10.0.0.20
  3. [decisao_monitorar] ameaca vazamento_dados em 10.0.0.20 sem evidencia de comprometimento de raiz

ISOLAR 10.0.0.66
  1. [indicador_integridade] checksum de /etc/shadow divergiu, alteracao ligada a 10.0.0.66
  2. [indicador_varredura] 512 portas distintas tocadas por 10.0.0.66
  3. [indicador_forca_bruta] 47 falhas de login vindas de 10.0.0.66
  4. [ameaca_intrusao] varredura e forca bruta partindo do mesmo host 10.0.0.66
  5. [ameaca_integridade] integridade violada em 10.0.0.66, sem outro indicador corroborando
  6. [ameaca_critica] integridade violada somada a intrusao_ativa no host 10.0.0.66
  7. [decisao_isolar] ameaca critica comprometimento_raiz confirmada em 10.0.0.66


## Autoverificacao

In [10]:
_autoverificar()

autoverificacao ok


## Controlador fuzzy Mamdani

A mesma evidencia bruta que alimenta o motor de regras acima — `portas`,
`tentativas_login`, `bytes_saida`, `hora` de cada `Conexao`, e `hash_mudou`
de cada `Arquivo` — tambem sustenta uma leitura continua de prioridade, em
vez de so uma decisao binaria. Em vez de inventar entradas novas, as duas
entradas do controlador (`severidade`, `confianca`) sao **calculadas a
partir dessa mesma evidencia**, reaproveitando os limiares que ja existiam:
`EXPEDIENTE` e `LIMITE_EXFILTRACAO`, e os mesmos cortes que as regras
`indicador_forca_bruta` (10 falhas) e `indicador_varredura` (100 portas) ja
usavam.

In [11]:
!pip install -q scikit-fuzzy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


### De evidencia crua a `severidade` e `confianca`

`severidade` (0 a 10) pesa o quanto cada indicador de v1 se aproxima do seu
proprio limiar, com o mesmo peso maior para `exfiltracao` e
`integridade_violada` que o README ja da a eles: um unico indicador desses
dois basta para sustentar uma ameaca de nivel alto em v1 (`ameaca_vazamento` e
`ameaca_integridade`), entao pesam mais aqui tambem.

Ate o limiar a contribuicao cresce linearmente (0 a 1). Acima dele ela **nao
satura**: cresce em escala logaritmica, +0.5 por decada (`_ALCANCE`). 512
portas pesam mais que 100, mas nao 5 vezes mais; so o total e cortado em 10.

`confianca` (0 a 100) soma um peso fixo por indicador de v1 que a evidencia
dispara. Os pesos nao sao uniformes pela mesma razao do texto de v1:
`varredura` e `forca_bruta` sozinhos sao fracos (so viram ameaca combinados,
na regra `ameaca_intrusao`), enquanto `exfiltracao` e `integridade_violada`
sozinhos ja bastam.

In [ ]:
import math

SEVERIDADE_PESO = {"varredura": 3, "forca_bruta": 3, "exfiltracao": 4, "integridade_violada": 4}
CONFIANCA_PESO = {"varredura": 15, "forca_bruta": 15, "exfiltracao": 40, "integridade_violada": 40}
_ALCANCE = 0.5  # quanto cada decada acima do limiar soma a um termo


def _escala(valor, limiar):
    """Linear ate o limiar (0 a 1), logaritmica acima dele, sem teto."""
    razao = valor / limiar
    return razao if razao <= 1 else 1 + _ALCANCE * math.log10(razao)


def severidade_de(portas, tentativas_login, bytes_saida, hora, hash_mudou):
    """Reaproveita os limiares de v1 (100 portas, 10 falhas, LIMITE_EXFILTRACAO)."""
    fora_de_hora = hora not in EXPEDIENTE
    termos = {
        "varredura": _escala(portas, 100),
        "forca_bruta": _escala(tentativas_login, 10),
        "exfiltracao": _escala(bytes_saida, LIMITE_EXFILTRACAO) if fora_de_hora else 0,
        "integridade_violada": 1 if hash_mudou else 0,
    }
    return min(sum(SEVERIDADE_PESO[k] * v for k, v in termos.items()), 10)


def confianca_de(portas, tentativas_login, bytes_saida, hora, hash_mudou):
    """Mesmos gatilhos das regras indicador_* de v1, um peso fixo por indicador disparado."""
    fora_de_hora = hora not in EXPEDIENTE
    disparados = {
        "varredura": portas >= 100,
        "forca_bruta": tentativas_login >= 10,
        "exfiltracao": bytes_saida > LIMITE_EXFILTRACAO and fora_de_hora,
        "integridade_violada": hash_mudou,
    }
    return min(sum(CONFIANCA_PESO[k] for k, v in disparados.items() if v), 100)

### As tres variaveis fuzzy

`severidade` e `confianca` tem tres termos (`baixa`, `media`, `alta`).
`prioridade` tem um quarto, `nula`, um ombro estreito em zero: sem ele, todo
host, mesmo o benigno, recebia ao menos `MONITORAR`, porque a saida nao tinha
como dizer "nenhuma acao". Funcoes de pertinencia triangulares (`trimf`),
cobrindo o dominio inteiro sem lacunas.

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

severidade = ctrl.Antecedent(np.arange(0, 10.01, 0.1), "severidade")
confianca = ctrl.Antecedent(np.arange(0, 100.01, 1), "confianca")
prioridade = ctrl.Consequent(np.arange(0, 10.01, 0.1), "prioridade")

severidade["baixa"] = fuzz.trimf(severidade.universe, [0, 0, 5])
severidade["media"] = fuzz.trimf(severidade.universe, [0, 5, 10])
severidade["alta"] = fuzz.trimf(severidade.universe, [5, 10, 10])

confianca["baixa"] = fuzz.trimf(confianca.universe, [0, 0, 50])
confianca["media"] = fuzz.trimf(confianca.universe, [0, 50, 100])
confianca["alta"] = fuzz.trimf(confianca.universe, [50, 100, 100])

prioridade["nula"] = fuzz.trimf(prioridade.universe, [0, 0, 2])
prioridade["baixa"] = fuzz.trimf(prioridade.universe, [0, 0, 5])
prioridade["media"] = fuzz.trimf(prioridade.universe, [0, 5, 10])
prioridade["alta"] = fuzz.trimf(prioridade.universe, [5, 10, 10])

severidade.view()
confianca.view()
prioridade.view()

### Base de regras

9 regras, uma para cada combinacao das 3x3 possibilidades de `severidade` x
`confianca`, entao nenhum ponto do espaco de entrada fica sem regra que o
cubra. O unico canto que vira `nula` e o de sinal e certeza ambos baixos:

| severidade \ confianca | baixa | media | alta |
| --- | --- | --- | --- |
| **baixa** | nula | baixa | media |
| **media** | baixa | media | alta |
| **alta** | media | alta | alta |

In [ ]:
regras = [
    ctrl.Rule(severidade["baixa"] & confianca["baixa"], prioridade["nula"]),
    ctrl.Rule(severidade["baixa"] & confianca["media"], prioridade["baixa"]),
    ctrl.Rule(severidade["baixa"] & confianca["alta"], prioridade["media"]),
    ctrl.Rule(severidade["media"] & confianca["baixa"], prioridade["baixa"]),
    ctrl.Rule(severidade["media"] & confianca["media"], prioridade["media"]),
    ctrl.Rule(severidade["media"] & confianca["alta"], prioridade["alta"]),
    ctrl.Rule(severidade["alta"] & confianca["baixa"], prioridade["media"]),
    ctrl.Rule(severidade["alta"] & confianca["media"], prioridade["alta"]),
    ctrl.Rule(severidade["alta"] & confianca["alta"], prioridade["alta"]),
]

sistema = ctrl.ControlSystem(regras)


def priorizar(sev, conf):
    sim = ctrl.ControlSystemSimulation(sistema)
    sim.input["severidade"] = sev
    sim.input["confianca"] = conf
    sim.compute()
    return sim.output["prioridade"]

### Quatro acoes: `NENHUMA`, `MONITORAR`, `RESTRINGIR`, `ISOLAR`

`ISOLAR`/`MONITORAR` sao as unicas acoes em v1 porque o motor de regras so
tem decisoes binarias. O controlador fuzzy ja calcula uma prioridade
continua; aproveitar isso para decidir entre mais acoes, e o que justifica
ter trazido Mamdani para o problema.

A acao nao vem de reler o numero defuzzificado (`sim.output["prioridade"]`)
contra limiares novos — isso seria refuzzificar um valor que ja perdeu
informacao no processo de centroide. Em vez disso, cada termo de
`prioridade` guarda sua propria forca de disparo agregada em
`prioridade.terms[label].membership_value[sim]`, e a acao e a do termo com
maior forca, a leitura mais direta que o Mamdani oferece. Por isso a acao e
o numero impresso ao lado dela podem discordar (ver o caso de intrusao forte
no grid): sao duas leituras da mesma agregacao.

| Termo vencedor | Acao |
| --- | --- |
| `nula` | `NENHUMA` (equivale a "sem decisao" de v1) |
| `baixa` | `MONITORAR` |
| `media` | `RESTRINGIR` (ex.: segmentar o host, revogar credenciais, sem isolar) |
| `alta` | `ISOLAR` |

Em empate, a ordem de leitura (`alta`, `media`, `baixa`, `nula`) favorece a
acao mais cautelosa, pelo mesmo motivo que v1 favorece escalada em
`ameaca_critica`.

In [ ]:
_ACAO_POR_TERMO = {"alta": "ISOLAR", "media": "RESTRINGIR", "baixa": "MONITORAR", "nula": "NENHUMA"}


def acao_fuzzy(sev, conf):
    sim = ctrl.ControlSystemSimulation(sistema)
    sim.input["severidade"] = sev
    sim.input["confianca"] = conf
    sim.compute()
    pertinencias = {t: prioridade.terms[t].membership_value[sim] for t in ("alta", "media", "baixa", "nula")}
    termo = max(pertinencias, key=pertinencias.get)
    return _ACAO_POR_TERMO[termo], sim.output["prioridade"], pertinencias

## Crisp (v1) x fuzzy (v2), lado a lado

O mesmo cenario de `demo()` — os tres hosts declarados na secao "Cenario da
memoria de trabalho", roda pelas duas leituras: a decisao do motor de regras
que ja tinha sido tomada em `motor`, e a acao de quatro vias que o controlador
fuzzy deriva da mesma evidencia bruta.

In [ ]:
_evidencia = {
    "10.0.0.66": dict(portas=512, tentativas_login=47, bytes_saida=1_000, hora=3, hash_mudou=True),
    "10.0.0.20": dict(portas=3, tentativas_login=0, bytes_saida=900_000_000, hora=2, hash_mudou=False),
    "10.0.0.5": dict(portas=2, tentativas_login=1, bytes_saida=1_000, hora=14, hash_mudou=False),
}
_decisoes_crisp = _acoes_por_alvo(motor)

print(f"{'host':<12} {'decisao crisp (v1)':<20} {'severidade':<11} {'confianca':<10} {'prioridade':<11} acao fuzzy (v2)")
for host, ev in _evidencia.items():
    sev = severidade_de(ev["portas"], ev["tentativas_login"], ev["bytes_saida"], ev["hora"], ev["hash_mudou"])
    conf = confianca_de(ev["portas"], ev["tentativas_login"], ev["bytes_saida"], ev["hora"], ev["hash_mudou"])
    acao, prio, _ = acao_fuzzy(sev, conf)
    decisao = _decisoes_crisp.get(host, "NENHUMA")
    print(f"{host:<12} {decisao:<20} {sev:<11.2f} {conf:<10.1f} {prio:<11.2f} {acao}")

host         decisao crisp (v1)   severidade  confianca  prioridade  acao fuzzy (v2)
10.0.0.66    ISOLAR               10.00       70.0       8.14        ISOLAR
10.0.0.20    MONITORAR            4.60        40.0       4.90        RESTRINGIR
10.0.0.5     NENHUMA              0.36        0.0        1.14        NENHUMA


## Autoverificacao fuzzy

Mesma logica de v1: extremos opostos do dominio produzem prioridades nos
extremos opostos, o ponto central fica perto do meio, a ordem das
prioridades concorda com a severidade das decisoes crisp (`ISOLAR` >
`MONITORAR` > sem decisao), e a acao de quatro vias reproduz essa mesma ordem
nos tres hosts do cenario: `ISOLAR`, `RESTRINGIR`, `NENHUMA`. O host
`10.0.0.20` e `RESTRINGIR` em v2 mas `MONITORAR` em v1: a divergencia e a
terceira acao em uso, e fica fixada no grid logo abaixo.

In [ ]:
def _autoverificar_fuzzy():
    baixa = priorizar(0, 0)
    alta = priorizar(10, 100)
    media = priorizar(5, 50)

    assert baixa < 3, baixa
    assert alta > 7, alta
    assert 3 <= media <= 7, media
    assert alta > baixa

    prio_isolar = priorizar(
        severidade_de(**_evidencia["10.0.0.66"]), confianca_de(**_evidencia["10.0.0.66"])
    )
    prio_monitorar = priorizar(
        severidade_de(**_evidencia["10.0.0.20"]), confianca_de(**_evidencia["10.0.0.20"])
    )
    prio_sem_decisao = priorizar(
        severidade_de(**_evidencia["10.0.0.5"]), confianca_de(**_evidencia["10.0.0.5"])
    )
    assert prio_isolar > prio_monitorar > prio_sem_decisao, (
        prio_isolar, prio_monitorar, prio_sem_decisao
    )

    acao_66, _, _ = acao_fuzzy(severidade_de(**_evidencia["10.0.0.66"]), confianca_de(**_evidencia["10.0.0.66"]))
    acao_20, _, _ = acao_fuzzy(severidade_de(**_evidencia["10.0.0.20"]), confianca_de(**_evidencia["10.0.0.20"]))
    acao_5, _, _ = acao_fuzzy(severidade_de(**_evidencia["10.0.0.5"]), confianca_de(**_evidencia["10.0.0.5"]))
    assert acao_66 == "ISOLAR", acao_66
    assert acao_20 == "RESTRINGIR", acao_20
    assert acao_5 == "NENHUMA", acao_5

    print("autoverificacao fuzzy ok")


_autoverificar_fuzzy()

autoverificacao fuzzy ok


## Grid de fronteiras: 5 casos

Cada caso e uma evidencia de um unico host, escolhida para cair exatamente
em cima de uma fronteira ou de um estado. As duas colunas de resultado
(`esperado v1`, `esperado v2`) sao fixadas: se um limiar, um peso ou uma regra
mudar, o caso quebra. Onde v1 e v2 divergem de proposito, a divergencia e a
informacao: o fuzzy enxerga o que o corte crisp descarta.

| # | Caso | Fronteira / estado | v1 | v2 |
| --- | --- | --- | --- | --- |
| 1 | 500 MB + 1 byte as 18h | ultima hora do expediente | `NENHUMA` | `NENHUMA` |
| 2 | 500 MB + 1 byte as 19h | primeira hora fora do expediente | `MONITORAR` | `RESTRINGIR` |
| 3 | 99 portas, 9 falhas | logo abaixo dos limiares | `NENHUMA` | `MONITORAR` |
| 4 | 100 portas, 10 falhas | exatamente nos limiares | `MONITORAR` | `RESTRINGIR` |
| 5 | 512 portas, 47 falhas, `/etc/shadow` alterado | escalada completa | `ISOLAR` | `ISOLAR` |

Os quatro estados (`NENHUMA`, `MONITORAR`, `RESTRINGIR`, `ISOLAR`) aparecem
em pelo menos um caso. Um arquivo critico alterado sozinho, sem outro sinal, e
coberto por `_verificar_integridade_isolada`.

In [ ]:
_SEM_SINAL = dict(portas=1, tentativas_login=0, bytes_saida=0, hora=14, hash_mudou=False)

# (nome, evidencia, esperado v1, esperado v2)
_GRID = [
    ("18h, volume alto", {**_SEM_SINAL, "bytes_saida": LIMITE_EXFILTRACAO + 1, "hora": 18}, "NENHUMA", "NENHUMA"),
    ("19h, volume alto", {**_SEM_SINAL, "bytes_saida": LIMITE_EXFILTRACAO + 1, "hora": 19}, "MONITORAR", "RESTRINGIR"),
    ("99p, 9 falhas", {**_SEM_SINAL, "portas": 99, "tentativas_login": 9}, "NENHUMA", "MONITORAR"),
    ("100p, 10 falhas", {**_SEM_SINAL, "portas": 100, "tentativas_login": 10}, "MONITORAR", "RESTRINGIR"),
    ("intrusao + shadow", {**_SEM_SINAL, "portas": 512, "tentativas_login": 47, "hora": 3, "hash_mudou": True}, "ISOLAR", "ISOLAR"),
]


def acao_crisp(ev):
    """Roda v1 sobre a evidencia de um unico host e devolve a acao (ou NENHUMA)."""
    m = Warden()
    m.reset()
    m.declare(Conexao(origem="9.9.9.9", **{k: v for k, v in ev.items() if k != "hash_mudou"}))
    if ev["hash_mudou"]:
        m.declare(Arquivo(caminho="/etc/shadow", origem="9.9.9.9", critico=True, hash_mudou=True))
    m.run()
    return _acoes_por_alvo(m).get("9.9.9.9", "NENHUMA")


print(f"{'caso':<20} {'sev':>5} {'conf':>5} {'prio':>5}  {'v1':<10} v2")
for nome, ev, esperado_v1, esperado_v2 in _GRID:
    sev, conf = severidade_de(**ev), confianca_de(**ev)
    acao, prio, _ = acao_fuzzy(sev, conf)
    v1 = acao_crisp(ev)
    print(f"{nome:<20} {sev:5.2f} {conf:5.1f} {prio:5.2f}  {v1:<10} {acao}")
    assert (v1, acao) == (esperado_v1, esperado_v2), (nome, v1, acao)

caso                   sev  conf  prio  v1         v2
18h, volume alto      0.03   0.0  0.72  NENHUMA    NENHUMA
19h, volume alto      4.03  40.0  4.90  MONITORAR  RESTRINGIR
99p, 9 falhas         5.67   0.0  2.87  NENHUMA    MONITORAR
100p, 10 falhas       6.00  30.0  4.73  MONITORAR  RESTRINGIR
intrusao + shadow    10.00  70.0  8.14  ISOLAR     ISOLAR


### Monotonicidade

Propriedade central de qualquer controlador de prioridade: mais severidade, ou
mais confianca, nunca derruba a prioridade. Varre as duas entradas em passos
de 1 e 5, uma de cada vez, com a outra fixa.

In [ ]:
def _verificar_monotonicidade():
    por_severidade = [priorizar(s, 40) for s in range(0, 11)]
    por_confianca = [priorizar(5, c) for c in range(0, 101, 5)]
    for nome, serie in (("severidade", por_severidade), ("confianca", por_confianca)):
        queda = [(a, b) for a, b in zip(serie, serie[1:]) if b < a - 1e-9]
        assert not queda, (nome, queda)
    print("monotonicidade ok")


_verificar_monotonicidade()

monotonicidade ok
